# Early Warning Tabular Experiments

Ce notebook reprend le script `early_warning_tabular_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Baseline physique/tabulaire causale pour l'alerte precoce en live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Tabular physics baselines for causal early warning.
- Run par defaut : `runs/exp_097_tabular_early_warning`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "early_warning_tabular_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import math
import time
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from ml_pipeline import ROOT, alarm_episodes, draw_overlay, load_dataset, load_json, read_frame, safe_auc, write_json, zone_polygon
from sequence_experiments import make_run_dir


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `build_target`

Cette cellule definit `build_target`. Elle prepare une partie du script.

In [ ]:
def build_target(df, min_early_s, max_early_s, mode):
    tte = pd.to_numeric(df["time_to_target_s"], errors="coerce")
    is_entry = df["is_danger_clip"].astype(int).eq(1)
    y = np.zeros(len(df), dtype=np.int32)
    usable = np.ones(len(df), dtype=bool)
    positive = is_entry & tte.ge(min_early_s) & tte.le(max_early_s)
    y[positive.to_numpy()] = 1
    too_late = is_entry & tte.ge(0.0) & tte.lt(min_early_s)
    if mode == "ignore_too_late":
        usable[too_late.to_numpy()] = False
    elif mode == "late_as_negative":
        pass
    else:
        raise ValueError(mode)
    return y, usable


## Fonction `add_physics_features`

Cette cellule definit `add_physics_features`. Elle prepare une partie du script.

In [ ]:
def add_physics_features(df):
    out = df.copy()
    eps = 1e-4
    if "min_signed_dist_norm" in out.columns:
        dist = out["min_signed_dist_norm"].astype(float)
    elif "max_signed_dist_norm" in out.columns:
        dist = out["max_signed_dist_norm"].astype(float)
    else:
        dist = pd.Series(np.zeros(len(out)), index=out.index)
    vel_cols = [c for c in out.columns if c.endswith("_signed_dist_vel")]
    if vel_cols:
        closing = -out[vel_cols].min(axis=1).astype(float)
    elif "max_signed_dist_vel" in out.columns:
        closing = -out["max_signed_dist_vel"].astype(float)
    else:
        closing = pd.Series(np.zeros(len(out)), index=out.index)
    out["physics_closing_speed"] = closing.replace([np.inf, -np.inf], 0).fillna(0)
    out["physics_ttz"] = (dist.clip(lower=0) / (closing.clip(lower=eps))).clip(0, 10).replace([np.inf, -np.inf], 10).fillna(10)
    out["physics_fast_approach"] = ((out["physics_ttz"] <= 1.0) & (out["physics_closing_speed"] > 0)).astype(int)
    return out


## Fonction `make_models`

Cette cellule definit `make_models`. Elle prepare une partie du script.

In [ ]:
def make_models(seed):
    return {
        "xgb_early": XGBClassifier(
            n_estimators=350,
            max_depth=2,
            learning_rate=0.025,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.25,
            reg_lambda=4.0,
            min_child_weight=3.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=seed,
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=4,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        ),
        "rf_balanced": RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=4,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        ),
        "logreg_l2": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", C=0.5, max_iter=2000, random_state=seed)),
    }


## Fonction `predict_positive`

Cette cellule definit `predict_positive`. Elle prepare une partie du script.

In [ ]:
def predict_positive(model, X):
    proba = model.predict_proba(X)
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "named_steps"):
        classes = model.named_steps[list(model.named_steps.keys())[-1]].classes_
    idx = list(classes).index(1) if classes is not None and 1 in list(classes) else 1
    return proba[:, idx]


## Fonction `alarm_episodes_hysteresis`

Cette cellule definit `alarm_episodes_hysteresis`. Elle prepare une partie du script.

In [ ]:
def alarm_episodes_hysteresis(times, scores, on_threshold, off_threshold, persistence_windows):
    alarms = []
    pending = []
    active = False
    for t, score in zip(times, scores):
        t = float(t)
        score = float(score)
        if active:
            if score < off_threshold:
                active = False
            continue
        if score >= on_threshold:
            pending.append(t)
            if len(pending) >= persistence_windows:
                alarms.append(pending[0])
                active = True
                pending = []
        else:
            pending = []
    if not alarms:
        return []
    episodes = [alarms[0]]
    for t in alarms[1:]:
        if t - episodes[-1] > 1.0:
            episodes.append(t)
    return episodes


## Fonction `evaluate`

Cette cellule definit `evaluate`. Elle prepare une partie du script.

In [ ]:
def evaluate(pred, threshold, split_name, early_margin_s, persistence_windows, hysteresis=False):
    sdf = pred[pred["split"].eq(split_name)].copy()
    pre = early = fp = danger = 0
    neg_min = 0.0
    early_times = []
    for _, g in sdf.groupby("video_id", sort=False):
        g = g.sort_values("time_s")
        if hysteresis:
            alarms = alarm_episodes_hysteresis(g["time_s"], g["risk"], threshold, max(0.01, threshold * 0.60), persistence_windows)
        else:
            alarms = alarm_episodes(g["time_s"], g["risk"], threshold, gap_s=1.0, persistence_windows=persistence_windows)
        is_danger = int(g["is_danger_clip"].max()) == 1
        target_values = pd.to_numeric(g["target_time_s"], errors="coerce").dropna()
        target = float(target_values.iloc[0]) if len(target_values) else math.nan
        if is_danger and not math.isnan(target):
            danger += 1
            pre_alarms = [float(t) for t in alarms if float(t) < target]
            if pre_alarms:
                first = min(pre_alarms)
                lead = target - first
                early_times.append(lead)
                pre += 1
                if lead >= early_margin_s:
                    early += 1
        else:
            fp += len(alarms)
            if len(g):
                neg_min += max(0.0, float(g["time_s"].max() - g["time_s"].min())) / 60.0
    precision = pre / (pre + fp) if (pre + fp) else np.nan
    pre_recall = pre / danger if danger else np.nan
    early_recall = early / danger if danger else np.nan
    return {
        "split": split_name,
        "threshold": float(threshold),
        "hysteresis": bool(hysteresis),
        "danger_videos": int(danger),
        "pre_entry_detected": int(pre),
        "early_detected": int(early),
        "false_alarm_episodes": int(fp),
        "pre_entry_recall": float(pre_recall),
        "early_recall": float(early_recall),
        "event_precision": float(precision),
        "false_alarms_per_min": float(fp / neg_min) if neg_min > 0 else 0.0,
        "median_early_warning_s": float(np.median(early_times)) if early_times else np.nan,
    }


## Fonction `selection_score`

Cette cellule definit `selection_score`. Elle prepare une partie du script.

In [ ]:
def selection_score(row):
    return 2.0 * row["early_recall"] + 0.8 * row["pre_entry_recall"] + 0.8 * row["event_precision"] - 0.07 * min(row["false_alarms_per_min"], 20.0)


## Fonction `make_failure_screenshots`

Cette cellule definit `make_failure_screenshots`. Elle prepare une partie du script.

In [ ]:
def make_failure_screenshots(run_dir, pred, selected):
    videos, _, _, zones = load_dataset()
    video_by_id = {row["video_id"]: row for row in videos}
    polygon = zone_polygon(zones)
    out_dir = run_dir / "error_review" / "tabular_causal_failures"
    out_dir.mkdir(parents=True, exist_ok=True)
    threshold = float(selected["threshold"])
    hysteresis = bool(selected["hysteresis"])
    saved = 0
    for video_id, g in pred[pred["split"].eq("test")].groupby("video_id", sort=False):
        if saved >= 16:
            break
        g = g.sort_values("time_s")
        alarms = alarm_episodes_hysteresis(g["time_s"], g["risk"], threshold, max(0.01, threshold * 0.60), 2) if hysteresis else alarm_episodes(g["time_s"], g["risk"], threshold, gap_s=1.0, persistence_windows=2)
        is_danger = int(g["is_danger_clip"].max()) == 1
        tv = pd.to_numeric(g["target_time_s"], errors="coerce").dropna()
        target = float(tv.iloc[0]) if len(tv) else math.nan
        label = None
        t = None
        if is_danger and not math.isnan(target):
            pre = [float(a) for a in alarms if float(a) < target]
            if not pre:
                label, t = "CAUSAL MISS", target
            elif target - min(pre) < 0.5:
                label, t = "LATE <0.5s", min(pre)
        elif alarms:
            label, t = "FALSE ALARM", min(alarms)
        if label is None or video_id not in video_by_id:
            continue
        video = video_by_id[video_id]
        frame_idx = int(round(t * float(video["fps"])))
        frame = read_frame(ROOT / video["path"], frame_idx)
        if frame is None:
            continue
        nearest = g.iloc[(g["time_s"] - t).abs().argsort().iloc[0]]
        out = draw_overlay(frame, polygon, [label, f"risk={float(nearest['risk']):.3f} thr={threshold:.2f}", f"target={target:.2f}s alarm={t:.2f}s", video["path"]])
        cv2.imwrite(str(out_dir / f"{video_id}_{label.lower().replace(' ', '_').replace('<', 'lt')}.jpg"), out)
        saved += 1
    return saved


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    base_run = resolve(args.base_run)
    run_dir = make_run_dir(args.run_name)
    windows = pd.read_csv(base_run / "features" / "window_features.csv")
    features = load_json(base_run / "features" / "window_feature_columns.json")["feature_columns"]
    windows = add_physics_features(windows)
    for extra in ["physics_closing_speed", "physics_ttz", "physics_fast_approach"]:
        if extra not in features:
            features.append(extra)
    windows[features] = windows[features].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y, usable = build_target(windows, args.min_early_s, args.max_early_s, args.target_mode)
    entry = pd.read_csv(base_run / "features" / "entry_times.csv")[["video_id", "event_type"]]
    windows = windows.merge(entry, on="video_id", how="left")
    hard_neg = windows["event_type"].fillna("").eq("near_miss") | ((windows["is_danger_clip"].astype(int).eq(1)) & (pd.to_numeric(windows["time_to_target_s"], errors="coerce") > args.max_early_s))
    sample_weight = np.ones(len(windows), dtype=np.float32)
    sample_weight[hard_neg.to_numpy()] *= args.hard_negative_weight
    sample_weight[~usable] = 0.0

    train_mask = windows["split"].eq("train").to_numpy() & usable
    X_train = windows.loc[train_mask, features].to_numpy(dtype=np.float32)
    y_train = y[train_mask]
    w_train = sample_weight[train_mask]
    models = make_models(args.seed)
    thresholds = [round(x, 2) for x in np.arange(0.05, 1.0, 0.05)]
    all_metrics = []
    all_pred = []
    for name, model in models.items():
        print(f"training {name}")
        start = time.perf_counter()
        if name.startswith("xgb"):
            positives = max(1, int(y_train.sum()))
            negatives = max(1, int(len(y_train) - positives))
            model.set_params(scale_pos_weight=negatives / positives)
        try:
            model.fit(X_train, y_train, sample_weight=w_train)
        except (TypeError, ValueError):
            if hasattr(model, "named_steps") and "logisticregression" in model.named_steps:
                model.fit(X_train, y_train, logisticregression__sample_weight=w_train)
            else:
                model.fit(X_train, y_train)
        train_time = time.perf_counter() - start
        pred = windows.copy()
        pred["risk"] = predict_positive(model, pred[features].to_numpy(dtype=np.float32))
        pred["model"] = name
        pred.to_csv(run_dir / "features" / f"predictions_{name}.csv", index=False)
        joblib.dump({"model": model, "features": features}, run_dir / "models" / f"{name}.joblib")
        all_pred.append(pred)
        for split in ["train", "val", "test"]:
            sdf = pred[pred["split"].eq(split)]
            ap = safe_auc(average_precision_score, y[pred["split"].eq(split).to_numpy()], sdf["risk"].to_numpy())
            auc = safe_auc(roc_auc_score, y[pred["split"].eq(split).to_numpy()], sdf["risk"].to_numpy())
            for threshold in thresholds:
                for hysteresis in [False, True]:
                    row = evaluate(pred, threshold, split, args.early_margin_s, args.persistence_windows, hysteresis)
                    row.update({"model": name, "average_precision": ap, "roc_auc": auc, "train_time_s": train_time, "selection_score": selection_score(row)})
                    all_metrics.append(row)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "tabular_early_warning_metrics.csv", index=False)
    val = metrics[metrics["split"].eq("val")].copy()
    best = val.sort_values("selection_score", ascending=False).iloc[0].to_dict()
    test = metrics[
        metrics["split"].eq("test")
        & metrics["model"].eq(best["model"])
        & np.isclose(metrics["threshold"].astype(float), float(best["threshold"]))
        & metrics["hysteresis"].eq(bool(best["hysteresis"]))
    ].iloc[0].to_dict()
    write_json(run_dir / "metrics" / "tabular_early_warning_best_selection.json", {"validation": best, "test": test})
    best_pred = next(p for p in all_pred if p["model"].iloc[0] == best["model"])
    saved = make_failure_screenshots(run_dir, best_pred, best)
    write_json(
        run_dir / "metrics" / "tabular_early_target_audit.json",
        {
            "rows": int(len(windows)),
            "usable_train_rows": int(train_mask.sum()),
            "positive_rows": int(y.sum()),
            "hard_negative_rows": int(hard_neg.sum()),
            "feature_count": int(len(features)),
        },
    )
    lines = ["# Tabular Physics Early-Warning Experiment", ""]
    lines.append("This trains engineered pose/geometry models directly for pre-entry warning using window-level physics features.")
    lines.append("")
    lines.append("| selected model | threshold | hysteresis | pre-entry recall | early recall | precision | FA/min | median early s | detected | early | danger |")
    lines.append("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
    lines.append(
        f"| {test['model']} | {float(test['threshold']):.2f} | {bool(test['hysteresis'])} | "
        f"{test['pre_entry_recall']:.3f} | {test['early_recall']:.3f} | {test['event_precision']:.3f} | "
        f"{test['false_alarms_per_min']:.3f} | {test['median_early_warning_s']:.3f} | {int(test['pre_entry_detected'])} | {int(test['early_detected'])} | {int(test['danger_videos'])} |"
    )
    lines.append("")
    lines.append("## Top Test Rows")
    lines.append("")
    lines.append("| model | threshold | hysteresis | pre-entry recall | early recall | precision | FA/min |")
    lines.append("|---|---:|---:|---:|---:|---:|---:|")
    view = metrics[metrics["split"].eq("test")].sort_values(["early_recall", "false_alarms_per_min", "event_precision"], ascending=[False, True, False]).head(20)
    for _, row in view.iterrows():
        lines.append(f"| {row['model']} | {row['threshold']:.2f} | {bool(row['hysteresis'])} | {row['pre_entry_recall']:.3f} | {row['early_recall']:.3f} | {row['event_precision']:.3f} | {row['false_alarms_per_min']:.3f} |")
    lines.append("")
    lines.append(f"Saved screenshots: `{saved}`")
    (run_dir / "tabular_early_warning_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(run_dir)
    print(run_dir / "tabular_early_warning_summary.md")


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Tabular physics baselines for causal early warning.")
    parser.add_argument("--base-run", default="runs/exp_070_physical_entry_baseline")
    parser.add_argument("--run-name", default="exp_097_tabular_early_warning")
    parser.add_argument("--target-mode", choices=["ignore_too_late", "late_as_negative"], default="ignore_too_late")
    parser.add_argument("--min-early-s", type=float, default=0.5)
    parser.add_argument("--max-early-s", type=float, default=1.5)
    parser.add_argument("--early-margin-s", type=float, default=0.5)
    parser.add_argument("--hard-negative-weight", type=float, default=2.5)
    parser.add_argument("--persistence-windows", type=int, default=2)
    parser.add_argument("--seed", type=int, default=777)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_097_tabular_early_warning_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["early_warning_tabular_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
